# 🐳 Clase 1: Introducción a Docker

---

> **Objetivo de la clase:** Comprender qué es Docker, por qué se usa, cómo funciona internamente y cómo comenzar a trabajar con él desde cero.

---

## Tabla de Contenidos

1. [¿Qué es Docker?](#1-qué-es-docker)
2. [Motivación: El problema del "en mi máquina funciona"](#2-motivación-el-problema-del-en-mi-máquina-funciona)
3. [Contenedores vs Máquinas Virtuales](#3-contenedores-vs-máquinas-virtuales)
4. [Arquitectura de Docker](#4-arquitectura-de-docker)
5. [Conceptos Fundamentales](#5-conceptos-fundamentales)
6. [Instalación de Docker](#6-instalación-de-docker)
7. [Primeros Pasos: Comandos Esenciales](#7-primeros-pasos-comandos-esenciales)
8. [Trabajando con Imágenes](#8-trabajando-con-imágenes)
9. [Trabajando con Contenedores](#9-trabajando-con-contenedores)
10. [Dockerfile: Construyendo tus propias imágenes](#10-dockerfile-construyendo-tus-propias-imágenes)
11. [Volúmenes y Persistencia de Datos](#11-volúmenes-y-persistencia-de-datos)
12. [Redes en Docker](#12-redes-en-docker)
13. [Docker Compose: Introducción](#13-docker-compose-introducción)
14. [Docker Hub y Registros](#14-docker-hub-y-registros)
15. [Buenas Prácticas](#15-buenas-prácticas)
16. [Referencias](#16-referencias)
17. [Temas para la Clase 2](#17-temas-para-la-clase-2)


---
## 1. ¿Qué es Docker?

**Docker** es una plataforma de código abierto que permite **construir, distribuir y ejecutar aplicaciones dentro de contenedores**. Fue lanzado en 2013 por Solomon Hykes en la empresa dotCloud (luego renombrada a Docker, Inc.) y desde entonces se ha convertido en el estándar de facto para la containerización de aplicaciones.

En términos sencillos:

> 🗃️ Docker empaqueta una aplicación y **todas sus dependencias** (librerías, configuraciones, variables de entorno, código) en una unidad estándar llamada **contenedor**, que puede ejecutarse en cualquier entorno de manera consistente.

### ¿Por qué es tan importante?

- **Portabilidad:** Un contenedor Docker se ejecuta igual en una laptop de desarrollo, en un servidor de pruebas y en producción en la nube.
- **Aislamiento:** Cada contenedor tiene su propio entorno, evitando conflictos entre dependencias.
- **Eficiencia:** Los contenedores son ligeros y arrancan en segundos.
- **Escalabilidad:** Facilita la orquestación y el escalado horizontal de aplicaciones.
- **Reproducibilidad:** El entorno de ejecución es exactamente el mismo siempre.

### Docker en cifras

| Dato | Valor |
|------|-------|
| Año de lanzamiento | 2013 |
| Repositorios en Docker Hub | +9 millones |
| Descargas de imágenes | +100 mil millones |
| Lenguaje principal | Go |
| Licencia | Apache 2.0 |


---
## 2. Motivación: El problema del "en mi máquina funciona"

Uno de los problemas más comunes en el desarrollo de software es el siguiente escenario:

```
Desarrollador: "El código funciona perfectamente en mi laptop."
Operaciones:   "En producción no arranca."
```

Esto ocurre por las **diferencias entre entornos de ejecución**:

- Versiones distintas de Python, Node.js, Java, etc.
- Librerías del sistema operativo diferentes (glibc, openssl, etc.)
- Variables de entorno configuradas de forma distinta.
- Rutas de archivos inconsistentes.
- Sistemas operativos completamente diferentes (Windows vs Linux).

### ¿Cómo resolvía esto la industria antes de Docker?

| Solución Anterior | Problema |
|-------------------|----------|
| Documentación de instalación | Se desactualiza, es propensa a errores humanos |
| Scripts de configuración (bash) | Frágiles, difíciles de mantener |
| Máquinas virtuales (VMs) | Pesadas (GBs), lentas de arrancar (minutos) |
| Configuración manual de servidores | No repetible, costosa |

### La solución de Docker

Docker introduce el concepto de **infraestructura como código** aplicado al entorno de ejecución:

```
Si funciona en el contenedor → funciona en CUALQUIER lugar que tenga Docker
```

El contenedor actúa como una **cápsula autocontenida** que lleva consigo todo lo necesario para ejecutar la aplicación, independientemente del sistema anfitrión.


---
## 3. Contenedores vs Máquinas Virtuales

Esta es quizás la comparación más importante para entender qué hace especial a Docker.

### Máquinas Virtuales (VMs)

```
┌───────────────────────────────────────────┐
│          Aplicación A   Aplicación B      │
├────────────────────┬──────────────────────┤
│   Guest OS (Linux) │  Guest OS (Windows)  │
├────────────────────┴──────────────────────┤
│              Hypervisor                   │
│     (VMware, VirtualBox, Hyper-V)         │
├───────────────────────────────────────────┤
│           Hardware físico / Host OS       │
└───────────────────────────────────────────┘
```

- Cada VM incluye un **sistema operativo completo** (varios GBs).
- El **hypervisor** virtualiza el hardware.
- Arranque lento: **minutos**.
- Alto consumo de recursos (CPU, RAM, disco).
- **Fuerte aislamiento** (kernel separado).

### Contenedores Docker

```
┌───────────────────────────────────────────┐
│  Contenedor A      Contenedor B           │
│  [App + Libs]      [App + Libs]           │
├───────────────────────────────────────────┤
│            Docker Engine                  │
├───────────────────────────────────────────┤
│          Sistema Operativo Host           │
├───────────────────────────────────────────┤
│              Hardware físico              │
└───────────────────────────────────────────┘
```

- Los contenedores **comparten el kernel** del sistema operativo anfitrión.
- Solo incluyen la aplicación y sus dependencias (MBs).
- Arranque ultra rápido: **milisegundos a segundos**.
- Mucho menor consumo de recursos.
- Aislamiento a nivel de procesos (namespaces + cgroups).

### Tabla comparativa

| Característica | Máquina Virtual | Contenedor Docker |
|---------------|-----------------|-------------------|
| Tamaño típico | 1–20 GB | 10–500 MB |
| Tiempo de arranque | 1–5 minutos | < 1 segundo |
| Aislamiento | Nivel de hardware (hypervisor) | Nivel de proceso (kernel) |
| Sistema operativo | Completo (por VM) | Compartido (host) |
| Portabilidad | Media | Alta |
| Overhead de rendimiento | Alto | Muy bajo (~5%) |
| Uso de memoria | Alto | Bajo |
| Casos de uso | Apps que requieren SO diferente, mayor aislamiento | Microservicios, CI/CD, dev local |

### ¿Son excluyentes?

No. En la práctica, **Docker corre dentro de VMs** en entornos de producción en la nube (AWS EC2, Google Compute Engine, etc.). La combinación aprovecha lo mejor de ambos mundos: el aislamiento de las VMs y la ligereza de los contenedores.


---
## 4. Arquitectura de Docker

Docker utiliza una arquitectura **cliente-servidor**. Sus componentes principales son:

```
┌──────────────────────────────────────────────────────┐
│                   CLIENTE DOCKER                     │
│  (CLI: docker build, docker run, docker pull, ...)   │
└───────────────────────┬──────────────────────────────┘
                        │  REST API / Unix socket
┌───────────────────────▼──────────────────────────────┐
│                  DOCKER DAEMON (dockerd)             │
│  - Gestiona imágenes, contenedores, redes, volúmenes │
│  - Escucha peticiones del cliente                    │
│  - Construye y ejecuta contenedores                  │
└───────────────────────┬──────────────────────────────┘
                        │
        ┌───────────────┴────────────────┐
        ▼                                ▼
┌───────────────┐              ┌─────────────────┐
│  containerd   │              │  Docker Registry │
│ (runtime de   │              │  (Docker Hub,    │
│ contenedores) │              │  ECR, GCR, etc.) │
└───────┬───────┘              └─────────────────┘
        │
┌───────▼───────┐
│   runc        │
│ (OCI runtime) │
└───────────────┘
```

### Componentes en detalle

#### 4.1 Docker Client (CLI)
La interfaz de línea de comandos que el usuario utiliza para interactuar con Docker. Cada comando `docker ...` envía una petición al daemon a través de la API REST.

```bash
# Ejemplos de comandos del cliente
docker run nginx
docker build -t mi-app .
docker ps
```

#### 4.2 Docker Daemon (`dockerd`)
El proceso principal que corre en segundo plano. Es el "cerebro" de Docker:
- Recibe comandos del cliente via API REST.
- Gestiona el ciclo de vida de contenedores e imágenes.
- Coordina el almacenamiento, redes y plugins.
- Delega la ejecución real de contenedores a `containerd`.

#### 4.3 containerd
Runtime de contenedores de nivel industrial. Gestiona:
- Ciclo de vida completo de contenedores (crear, ejecutar, pausar, eliminar).
- Descarga y almacenamiento de imágenes.
- Gestión de snapshots del sistema de archivos.

#### 4.4 runc
La implementación de referencia del estándar OCI (Open Container Initiative). Es el proceso que finalmente crea y ejecuta el contenedor a nivel de sistema operativo usando:
- **Linux Namespaces:** Aislamiento de PID, red, sistema de archivos, usuarios, etc.
- **cgroups (Control Groups):** Limitación de recursos (CPU, memoria, I/O).

#### 4.5 Docker Registry
Almacén de imágenes Docker. El registro público por defecto es **Docker Hub** (`hub.docker.com`), pero existen alternativas privadas:
- Amazon ECR (Elastic Container Registry)
- Google Container Registry (GCR)
- GitHub Container Registry (GHCR)
- Harbor (auto-hospedado)

### 4.6 Tecnologías del kernel que hacen posible los contenedores

| Tecnología | Función |
|------------|--------|
| **Linux Namespaces** | Aíslan el proceso: PID, red, montajes, IPC, UTS, usuario |
| **cgroups v2** | Limitan y monitorean recursos: CPU, RAM, I/O, red |
| **Union File Systems** | Capas de sistema de archivos (OverlayFS, AUFS) |
| **seccomp** | Filtrado de llamadas al sistema (seguridad) |
| **AppArmor/SELinux** | Políticas de seguridad adicionales |


---
## 5. Conceptos Fundamentales

### 5.1 Imagen (Image)

Una **imagen** es una plantilla de solo lectura que contiene:
- El sistema de archivos base (Ubuntu, Alpine, Debian, etc.)
- El código de la aplicación
- Las dependencias y librerías
- Las instrucciones de configuración

Las imágenes se construyen en **capas** (layers). Cada instrucción en un `Dockerfile` crea una nueva capa:

```
┌─────────────────────────────────┐  ← Capa 4: COPY app.py
├─────────────────────────────────┤  ← Capa 3: RUN pip install flask
├─────────────────────────────────┤  ← Capa 2: RUN apt-get update
├─────────────────────────────────┤  ← Capa 1 (base): FROM python:3.11
└─────────────────────────────────┘
```

Las capas son **inmutables** y se **reutilizan** entre imágenes (caché de construcción). Esto ahorra espacio y tiempo.

**Nomenclatura de imágenes:**
```
[registro/][usuario/]nombre[:tag]

Ejemplos:
  nginx                          → nginx:latest (Docker Hub oficial)
  python:3.11-slim               → imagen oficial con tag específico
  myuser/myapp:v1.0              → imagen de usuario en Docker Hub
  ghcr.io/myorg/myapp:sha-abc123 → imagen en GitHub Container Registry
```

### 5.2 Contenedor (Container)

Un **contenedor** es una **instancia en ejecución** de una imagen. Es a las imágenes lo que un proceso es a un programa ejecutable.

```
Imagen (plantilla, inmutable) → docker run → Contenedor (proceso en ejecución)
```

Características:
- Tiene su propia capa de escritura (writable layer) sobre las capas de la imagen.
- Todos los cambios en el sistema de archivos del contenedor se guardan en esa capa.
- Al eliminar el contenedor, esa capa se pierde (a menos que se use un volumen).
- Puede estar en estados: `created`, `running`, `paused`, `stopped`, `removed`.

### 5.3 Dockerfile

Un **Dockerfile** es un archivo de texto con instrucciones para construir una imagen de forma automatizada y reproducible:

```dockerfile
FROM python:3.11-slim        # Imagen base
WORKDIR /app                  # Directorio de trabajo
COPY requirements.txt .       # Copiar archivo de dependencias
RUN pip install -r requirements.txt  # Instalar dependencias
COPY . .                      # Copiar código fuente
EXPOSE 8080                   # Puerto expuesto
CMD ["python", "app.py"]      # Comando por defecto al ejecutar
```

### 5.4 Volumen (Volume)

Un **volumen** es el mecanismo oficial de Docker para **persistir datos** generados y utilizados por los contenedores. Los datos de un volumen sobreviven al ciclo de vida del contenedor.

```
Contenedor A ──┐
               ├── /data ←→ Volumen Docker (en el host: /var/lib/docker/volumes/...)
Contenedor B ──┘
```

### 5.5 Red (Network)

Docker provee **redes virtuales** para comunicar contenedores entre sí y con el exterior. Por defecto crea tres redes: `bridge`, `host` y `none`.

```
Contenedor Web ←──── red bridge ────→ Contenedor DB
      ↕
   Puerto 80 expuesto al Host
```

### 5.6 Docker Compose

Herramienta para definir y gestionar aplicaciones **multi-contenedor** mediante un archivo YAML (`docker-compose.yml`). Permite orquestar servicios, redes y volúmenes con un solo comando.

### 5.7 Docker Hub / Registry

Repositorio centralizado donde se almacenan y comparten imágenes Docker. Funciona de manera análoga a GitHub pero para imágenes de contenedores.


---
## 6. Instalación de Docker

### 6.1 Requisitos del sistema

| Sistema Operativo | Requisito |
|-------------------|-----------|
| Linux (Ubuntu/Debian/Fedora/RHEL) | Kernel 3.10+, 64 bits |
| macOS | macOS 12+ (Monterey), chip Intel o Apple Silicon |
| Windows | Windows 10/11 Pro/Enterprise/Education con WSL2 |

### 6.2 Instalación en Ubuntu/Debian

```bash
# 1. Actualizar el índice de paquetes
sudo apt-get update

# 2. Instalar dependencias
sudo apt-get install -y \
    ca-certificates \
    curl \
    gnupg

# 3. Agregar la clave GPG oficial de Docker
sudo install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | \
    sudo gpg --dearmor -o /etc/apt/keyrings/docker.gpg
sudo chmod a+r /etc/apt/keyrings/docker.gpg

# 4. Agregar el repositorio de Docker
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] \
  https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

# 5. Instalar Docker Engine
sudo apt-get update
sudo apt-get install -y docker-ce docker-ce-cli containerd.io \
    docker-buildx-plugin docker-compose-plugin

# 6. Agregar el usuario al grupo docker (evitar usar sudo)
sudo usermod -aG docker $USER
newgrp docker

# 7. Verificar instalación
docker --version
docker run hello-world
```

### 6.3 Instalación en macOS y Windows

La forma más sencilla en macOS y Windows es instalar **Docker Desktop**:

1. Descargar desde: https://www.docker.com/products/docker-desktop/
2. Ejecutar el instalador y seguir las instrucciones.
3. En Windows, habilitar **WSL2** (Windows Subsystem for Linux 2) cuando se solicite.

Docker Desktop incluye:
- Docker Engine
- Docker CLI
- Docker Compose
- Docker Scout (análisis de seguridad)
- Interfaz gráfica de gestión
- Integración con Kubernetes

### 6.4 Verificar la instalación

```bash
# Ver versión del cliente y servidor
docker version

# Ver información del sistema
docker info

# Ejecutar el contenedor de prueba oficial
docker run hello-world
```

La salida de `docker run hello-world` confirma que la instalación funciona correctamente y explica los pasos que Docker ejecutó para correr ese contenedor.


---
## 7. Primeros Pasos: Comandos Esenciales

La CLI de Docker sigue la estructura:
```
docker [opciones] COMANDO [argumentos]
docker [opciones] OBJETO COMANDO [argumentos]   ← nuevo estilo recomendado
```

### 7.1 Comando `docker run` — el más importante

`docker run` crea y arranca un contenedor desde una imagen:

```bash
# Sintaxis básica
docker run [opciones] IMAGEN [comando] [argumentos]

# Ejecutar un contenedor en primer plano
docker run ubuntu echo "Hola desde Docker"

# Ejecutar en modo interactivo (-it = interactive + pseudo-TTY)
docker run -it ubuntu bash

# Ejecutar en segundo plano (detached)
docker run -d nginx

# Mapear puerto del host → contenedor (host:contenedor)
docker run -d -p 8080:80 nginx

# Asignar un nombre al contenedor
docker run -d --name mi-nginx -p 8080:80 nginx

# Pasar variables de entorno
docker run -e MI_VARIABLE=valor ubuntu env

# Montar un directorio del host
docker run -v /ruta/host:/ruta/contenedor nginx

# Eliminar el contenedor al terminar (--rm)
docker run --rm ubuntu echo "Esto se elimina al terminar"

# Limitar recursos
docker run --memory="512m" --cpus="1.5" nginx
```

### 7.2 Resumen de opciones más usadas de `docker run`

| Opción | Descripción |
|--------|-------------|
| `-d` | Ejecutar en segundo plano (detached) |
| `-it` | Modo interactivo con terminal |
| `--rm` | Eliminar el contenedor al detenerse |
| `--name <nombre>` | Asignar nombre al contenedor |
| `-p <host>:<cont>` | Mapear puertos |
| `-v <host>:<cont>` | Montar volumen/directorio |
| `-e KEY=VALUE` | Variable de entorno |
| `--env-file <archivo>` | Cargar variables desde archivo |
| `--network <red>` | Conectar a una red específica |
| `--memory` | Límite de memoria RAM |
| `--cpus` | Límite de CPU |
| `--restart` | Política de reinicio (no, always, on-failure) |


---
## 8. Trabajando con Imágenes

### 8.1 Buscar imágenes

```bash
# Buscar en Docker Hub desde la CLI
docker search nginx
docker search --filter=is-official=true python
docker search --limit 5 ubuntu
```

También puedes buscar directamente en: https://hub.docker.com

### 8.2 Descargar imágenes

```bash
# Descargar la versión más reciente (latest)
docker pull nginx

# Descargar una versión específica
docker pull python:3.11-slim

# Descargar desde un registro diferente
docker pull ghcr.io/owner/image:tag

# Descargar para múltiples arquitecturas
docker pull --platform linux/amd64 nginx
```

### 8.3 Listar imágenes locales

```bash
# Listar todas las imágenes
docker images
docker image ls

# Mostrar solo IDs
docker images -q

# Ver imágenes con formato personalizado
docker images --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"

# Filtrar por nombre
docker images python
```

### 8.4 Inspeccionar imágenes

```bash
# Ver metadatos detallados de una imagen
docker inspect nginx

# Ver el historial de capas de una imagen
docker history nginx
docker history --no-trunc python:3.11-slim
```

### 8.5 Eliminar imágenes

```bash
# Eliminar una imagen por nombre o ID
docker rmi nginx
docker image rm python:3.11

# Forzar eliminación aunque haya contenedores basados en ella
docker rmi -f nginx

# Eliminar todas las imágenes no usadas (dangling)
docker image prune

# Eliminar todas las imágenes no referenciadas por ningún contenedor
docker image prune -a
```

### 8.6 Exportar e importar imágenes

```bash
# Exportar imagen a un archivo tar
docker save -o mi-imagen.tar nginx:latest

# Importar imagen desde un archivo tar
docker load -i mi-imagen.tar

# Etiquetar una imagen (crear alias)
docker tag nginx:latest mi-registro.com/mi-nginx:v1.0

# Subir imagen a un registro
docker push mi-usuario/mi-app:v1.0
```


---
## 9. Trabajando con Contenedores

### 9.1 Ciclo de vida de un contenedor

```
         docker create
              ↓
          [created]
              ↓
         docker start
              ↓
          [running] ←──── docker unpause
         ↙       ↘
  docker stop  docker pause
      ↓              ↓
  [stopped]       [paused]
      ↓
  docker rm
      ↓
  [removed]
```

### 9.2 Gestionar contenedores

```bash
# Listar contenedores en ejecución
docker ps
docker container ls

# Listar TODOS los contenedores (incluyendo detenidos)
docker ps -a

# Listar solo IDs
docker ps -q

# Iniciar un contenedor detenido
docker start mi-contenedor

# Detener un contenedor (señal SIGTERM, luego SIGKILL tras timeout)
docker stop mi-contenedor

# Forzar detención inmediata (SIGKILL)
docker kill mi-contenedor

# Reiniciar un contenedor
docker restart mi-contenedor

# Pausar/reanudar
docker pause mi-contenedor
docker unpause mi-contenedor

# Eliminar un contenedor detenido
docker rm mi-contenedor

# Forzar eliminación de contenedor en ejecución
docker rm -f mi-contenedor

# Eliminar todos los contenedores detenidos
docker container prune
```

### 9.3 Monitoreo e inspección

```bash
# Ver logs de un contenedor
docker logs mi-contenedor

# Seguir logs en tiempo real
docker logs -f mi-contenedor

# Ver últimas N líneas
docker logs --tail 50 mi-contenedor

# Ver estadísticas en tiempo real (CPU, mem, net, I/O)
docker stats
docker stats mi-contenedor

# Ver procesos dentro del contenedor
docker top mi-contenedor

# Inspeccionar metadatos del contenedor (JSON completo)
docker inspect mi-contenedor

# Ver diferencias en el sistema de archivos
docker diff mi-contenedor

# Ver los puertos mapeados
docker port mi-contenedor
```

### 9.4 Ejecutar comandos en contenedores en ejecución

```bash
# Ejecutar un comando en un contenedor corriendo
docker exec mi-contenedor ls /app

# Abrir una shell interactiva en el contenedor
docker exec -it mi-contenedor bash
docker exec -it mi-contenedor sh      # si bash no está disponible

# Ejecutar como usuario específico
docker exec -u root -it mi-contenedor bash

# Copiar archivos entre host y contenedor
docker cp archivo.txt mi-contenedor:/app/
docker cp mi-contenedor:/app/log.txt ./log.txt
```

### 9.5 Crear imagen desde un contenedor modificado

```bash
# Crear imagen a partir del estado actual de un contenedor
# (No es la práctica recomendada, pero es útil para debug)
docker commit mi-contenedor mi-nueva-imagen:v1
```


---
## 10. Dockerfile: Construyendo tus propias imágenes

Un `Dockerfile` es la receta para construir una imagen personalizada.

### 10.1 Instrucciones principales

| Instrucción | Descripción | Ejemplo |
|-------------|-------------|--------|
| `FROM` | Imagen base (siempre primera) | `FROM python:3.11-slim` |
| `LABEL` | Metadatos (autor, versión, etc.) | `LABEL maintainer="yo@mail.com"` |
| `RUN` | Ejecuta comando durante build (nueva capa) | `RUN apt-get install -y curl` |
| `COPY` | Copia archivos del host → imagen | `COPY ./src /app/src` |
| `ADD` | Como COPY pero acepta URLs y descomprime tarballs | `ADD app.tar.gz /app/` |
| `WORKDIR` | Establece directorio de trabajo | `WORKDIR /app` |
| `ENV` | Define variable de entorno | `ENV PORT=8080` |
| `ARG` | Argumento de build (no persiste en imagen) | `ARG VERSION=1.0` |
| `EXPOSE` | Documenta el puerto (no lo abre) | `EXPOSE 8080` |
| `VOLUME` | Crea un punto de montaje | `VOLUME ["/data"]` |
| `USER` | Usuario para ejecutar procesos siguientes | `USER appuser` |
| `CMD` | Comando por defecto al correr el contenedor | `CMD ["python", "app.py"]` |
| `ENTRYPOINT` | Punto de entrada del contenedor | `ENTRYPOINT ["gunicorn"]` |
| `HEALTHCHECK` | Chequeo de salud del contenedor | `HEALTHCHECK CMD curl -f http://localhost/` |
| `ONBUILD` | Instrucción que se ejecuta al hacer FROM de esta imagen | `ONBUILD COPY . /app` |

### 10.2 Diferencia entre CMD y ENTRYPOINT

```dockerfile
# CMD: el comando completo, se puede sobreescribir al hacer docker run
CMD ["python", "app.py"]
# → docker run mi-imagen  (ejecuta python app.py)
# → docker run mi-imagen bash  (sobreescribe, ejecuta bash)

# ENTRYPOINT: siempre se ejecuta, no se sobreescribe con argumentos
ENTRYPOINT ["python"]
CMD ["app.py"]  # argumento por defecto al entrypoint
# → docker run mi-imagen  (ejecuta python app.py)
# → docker run mi-imagen otro.py  (ejecuta python otro.py)
```

### 10.3 Ejemplo completo: Aplicación Python con Flask

Estructura del proyecto:
```
mi-app/
├── Dockerfile
├── requirements.txt
├── .dockerignore
└── app.py
```

**`app.py`:**
```python
from flask import Flask
app = Flask(__name__)

@app.route('/')
def hello():
    return '<h1>Hola desde Docker! 🐳</h1>'

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
```

**`requirements.txt`:**
```
flask==3.0.0
```

**`Dockerfile`:**
```dockerfile
# Imagen base oficial de Python (versión slim = más pequeña)
FROM python:3.11-slim

# Metadatos
LABEL maintainer="yo@ejemplo.com"
LABEL version="1.0"

# Establecer directorio de trabajo
WORKDIR /app

# Copiar e instalar dependencias primero (aprovechar caché)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar el código de la aplicación
COPY . .

# Crear usuario no privilegiado (seguridad)
RUN adduser --disabled-password --gecos '' appuser
USER appuser

# Exponer el puerto
EXPOSE 5000

# Comando de inicio
CMD ["python", "app.py"]
```

**`.dockerignore`:**
```
__pycache__/
*.pyc
*.pyo
.git/
.gitignore
.env
*.md
tests/
```

### 10.4 Construir y ejecutar

```bash
# Construir la imagen
# -t = tag (nombre:version), . = contexto de build (directorio actual)
docker build -t mi-app:1.0 .

# Construir sin caché
docker build --no-cache -t mi-app:1.0 .

# Construir pasando argumentos
docker build --build-arg VERSION=2.0 -t mi-app:2.0 .

# Ejecutar el contenedor
docker run -d -p 5000:5000 --name mi-flask-app mi-app:1.0

# Verificar que funciona
curl http://localhost:5000
```

### 10.5 Multi-stage builds (builds multietapa)

Técnica para reducir el tamaño de las imágenes finales usando múltiples etapas `FROM`:

```dockerfile
# Etapa 1: Builder (compila el código)
FROM golang:1.21 AS builder
WORKDIR /src
COPY . .
RUN go build -o mi-app .

# Etapa 2: Imagen final (solo el binario compilado)
FROM alpine:3.19
COPY --from=builder /src/mi-app /usr/local/bin/mi-app
ENTRYPOINT ["mi-app"]
```

Con esto, la imagen final no incluye el compilador de Go (que pesa ~800MB), resultando en una imagen de ~10MB.


---
## 11. Volúmenes y Persistencia de Datos

Por defecto, los datos dentro de un contenedor son **efímeros**: al eliminar el contenedor, los datos desaparecen. Los volúmenes solucionan este problema.

### 11.1 Tipos de almacenamiento en Docker

```
┌─────────────────────────────────────────────────────┐
│                    Contenedor                       │
│  /app (writable layer - temporal)                   │
│  /datos ────────────────────────────────────────┐   │
│  /config ──────────────────────────────────┐    │   │
└────────────────────────────────────────────┼────┼───┘
                                             │    │
                    ┌────────────────────────┘    │
                    │                             │
          ┌─────────▼──────────┐      ┌──────────▼────────┐
          │  Bind Mount        │      │  Volume de Docker  │
          │  (directorio host) │      │  (gestionado por   │
          │  /home/user/config │      │  Docker Engine)    │
          └────────────────────┘      └───────────────────-┘
```

| Tipo | Descripción | Caso de uso |
|------|-------------|-------------|
| **Volumes** | Gestionados por Docker, almacenados en `/var/lib/docker/volumes/` | Persistencia de datos de producción |
| **Bind mounts** | Directorio/archivo del host montado en el contenedor | Desarrollo local (hot reload) |
| **tmpfs mounts** | Almacenamiento solo en memoria RAM | Datos temporales sensibles |

### 11.2 Volumes (recomendados para producción)

```bash
# Crear un volumen
docker volume create mi-volumen

# Listar volúmenes
docker volume ls

# Inspeccionar un volumen
docker volume inspect mi-volumen

# Usar un volumen en un contenedor
docker run -d \
  --name mi-postgres \
  -v mi-volumen:/var/lib/postgresql/data \
  -e POSTGRES_PASSWORD=secreto \
  postgres:16

# Eliminar un volumen (debe estar sin uso)
docker volume rm mi-volumen

# Eliminar todos los volúmenes no usados
docker volume prune
```

### 11.3 Bind Mounts (ideal para desarrollo)

```bash
# Montar directorio del host en el contenedor
docker run -d \
  --name mi-dev \
  -v $(pwd):/app \
  -p 5000:5000 \
  mi-app:dev

# Con --mount (sintaxis más explícita y recomendada)
docker run -d \
  --mount type=bind,source=$(pwd),target=/app \
  -p 5000:5000 \
  mi-app:dev
```

### 11.4 Compartir volúmenes entre contenedores

```bash
# Contenedor 1: escribe datos
docker run -d --name escritor \
  -v datos-compartidos:/datos \
  ubuntu bash -c "while true; do date >> /datos/fecha.txt; sleep 1; done"

# Contenedor 2: lee los datos escritos por contenedor 1
docker run --rm \
  -v datos-compartidos:/datos:ro \
  ubuntu cat /datos/fecha.txt
# :ro = read-only (solo lectura)
```


---
## 12. Redes en Docker

Docker crea una abstracción de red que permite a los contenedores comunicarse entre sí y con el mundo exterior de forma controlada.

### 12.1 Redes por defecto

```bash
# Ver todas las redes
docker network ls
```

| Red | Driver | Descripción |
|-----|--------|-------------|
| `bridge` | bridge | Red por defecto para contenedores en el mismo host |
| `host` | host | Comparte la red del host (sin aislamiento) |
| `none` | null | Sin acceso a red |

### 12.2 Tipos de redes (drivers)

| Driver | Descripción | Uso |
|--------|-------------|-----|
| **bridge** | Red virtual NAT. Contenedores en la misma red bridge pueden comunicarse | Desarrollo local, apps multi-contenedor |
| **host** | Contenedor usa directamente la red del host | Alto rendimiento de red (Linux) |
| **overlay** | Red que abarca múltiples hosts Docker | Swarm / clústeres |
| **macvlan** | Asigna dirección MAC al contenedor (como dispositivo físico) | Apps legacy que necesitan IP en la red LAN |
| **none** | Sin ninguna red | Máxima seguridad, sin acceso de red |

### 12.3 Redes bridge personalizadas

Las redes bridge personalizadas son **superiores** a la red bridge por defecto porque:
- Proveen **resolución DNS por nombre de contenedor**.
- Permiten conectar/desconectar contenedores dinámicamente.
- Proporcionan mejor aislamiento.

```bash
# Crear una red personalizada
docker network create mi-red
docker network create --driver bridge --subnet 172.20.0.0/16 mi-red

# Conectar contenedores a la red
docker run -d --name backend --network mi-red mi-app-backend
docker run -d --name frontend --network mi-red mi-app-frontend

# Ahora el frontend puede alcanzar el backend por su nombre:
# http://backend:5000  → funciona gracias al DNS interno de Docker

# Conectar un contenedor existente a una red
docker network connect mi-red mi-contenedor

# Desconectar un contenedor de una red
docker network disconnect mi-red mi-contenedor

# Inspeccionar una red
docker network inspect mi-red

# Eliminar una red
docker network rm mi-red

# Eliminar redes sin uso
docker network prune
```

### 12.4 Exposición de puertos

```bash
# -p <puerto_host>:<puerto_contenedor>
docker run -p 8080:80 nginx     # mapea host:8080 → contenedor:80
docker run -p 443:443 nginx     # mapea mismo puerto
docker run -p 127.0.0.1:80:80 nginx  # solo en interfaz loopback del host

# -P (mayúscula): puertos aleatorios del host
docker run -P nginx
docker port mi-contenedor  # ver qué puertos se asignaron
```


---
## 13. Docker Compose: Introducción

**Docker Compose** es una herramienta que permite definir y ejecutar aplicaciones Docker **multi-contenedor** con un único archivo de configuración `docker-compose.yml` (o `compose.yml`).

### 13.1 ¿Por qué Docker Compose?

Sin Compose, para levantar una app con frontend + backend + base de datos necesitarías:

```bash
# Sin Docker Compose (tedioso)
docker network create app-network
docker volume create db-data
docker run -d --name db --network app-network -v db-data:/data -e POSTGRES_PASSWORD=secret postgres
docker run -d --name backend --network app-network -e DB_HOST=db -p 5000:5000 mi-backend
docker run -d --name frontend --network app-network -p 3000:3000 mi-frontend
```

Con Compose, todo esto se define en un archivo y se ejecuta con un solo comando:

```bash
docker compose up -d
```

### 13.2 Estructura del archivo `docker-compose.yml`

```yaml
# docker-compose.yml
# Versión del formato de Compose
# (en versiones modernas, 'version' es opcional)
version: '3.8'

services:           # Definición de los contenedores
  web:              # Nombre del servicio
    build: .        # Construir imagen desde el Dockerfile del directorio actual
    ports:
      - "5000:5000"
    depends_on:
      - db           # Arranca después del servicio 'db'
    environment:
      - DATABASE_URL=postgresql://user:secret@db:5432/mydb
    networks:
      - app-net

  db:
    image: postgres:16-alpine   # Usar imagen directamente (sin construir)
    volumes:
      - db-data:/var/lib/postgresql/data
    environment:
      POSTGRES_USER: user
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: mydb
    networks:
      - app-net

  redis:
    image: redis:7-alpine
    networks:
      - app-net

volumes:            # Definición de volúmenes
  db-data:

networks:           # Definición de redes
  app-net:
    driver: bridge
```

### 13.3 Comandos principales de Docker Compose

```bash
# Levantar todos los servicios (en segundo plano)
docker compose up -d

# Levantar y reconstruir imágenes
docker compose up -d --build

# Ver el estado de los servicios
docker compose ps

# Ver logs de todos los servicios
docker compose logs

# Ver logs de un servicio específico
docker compose logs -f web

# Detener los servicios (sin eliminar)
docker compose stop

# Detener y eliminar contenedores, redes (NO volúmenes)
docker compose down

# Detener y eliminar TODO (incluyendo volúmenes)
docker compose down -v

# Escalar un servicio
docker compose up -d --scale web=3

# Ejecutar un comando en un servicio
docker compose exec db psql -U user mydb
docker compose exec web bash

# Correr un servicio una sola vez (ephemeral)
docker compose run --rm web python manage.py migrate
```

### 13.4 Override files

Docker Compose permite múltiples archivos de configuración para diferentes entornos:

```bash
# Base + override de desarrollo
docker compose -f docker-compose.yml -f docker-compose.dev.yml up

# Base + override de producción
docker compose -f docker-compose.yml -f docker-compose.prod.yml up
```

Estructura típica:
```
docker-compose.yml          # Configuración base
docker-compose.dev.yml      # Overrides para desarrollo (hot reload, debug)
docker-compose.prod.yml     # Overrides para producción (réplicas, limits)
docker-compose.test.yml     # Overrides para testing
```


---
## 14. Docker Hub y Registros

### 14.1 Docker Hub

**Docker Hub** (https://hub.docker.com) es el registro de imágenes más grande del mundo. Contiene:

- **Imágenes oficiales:** Mantenidas por Docker y los proyectos oficiales. Ej: `ubuntu`, `nginx`, `python`, `postgres`, `redis`, `node`.
- **Imágenes verificadas de publishers:** Empresas como Microsoft, Oracle, etc.
- **Imágenes de la comunidad:** Publicadas por usuarios.

### 14.2 Autenticarse y publicar imágenes

```bash
# Iniciar sesión en Docker Hub
docker login
# Introduce usuario y contraseña/token

# Iniciar sesión en un registro privado
docker login mi-registro.empresa.com

# Etiquetar la imagen con el nombre del registro
docker tag mi-app:1.0 miusuario/mi-app:1.0
docker tag mi-app:1.0 miusuario/mi-app:latest

# Subir la imagen
docker push miusuario/mi-app:1.0
docker push miusuario/mi-app:latest

# Cerrar sesión
docker logout
```

### 14.3 Registro privado con Harbor o Docker Registry

```bash
# Levantar un registro Docker local (para pruebas)
docker run -d -p 5001:5000 --name mi-registro registry:2

# Etiquetar y subir al registro local
docker tag mi-app:1.0 localhost:5001/mi-app:1.0
docker push localhost:5001/mi-app:1.0

# Descargar del registro local
docker pull localhost:5001/mi-app:1.0
```

### 14.4 Estrategia de etiquetado (tagging)

Una buena estrategia de tags es fundamental para la gestión de versiones:

```bash
# Semantic versioning
mi-app:1.2.3
mi-app:1.2
mi-app:1
mi-app:latest

# Por commit SHA (ideal para CI/CD)
mi-app:sha-abc1234

# Por entorno
mi-app:dev
mi-app:staging
mi-app:prod

# Por fecha de build
mi-app:20241215
```


---
## 15. Buenas Prácticas

### 15.1 Buenas prácticas para Dockerfiles

#### ✅ Usar imágenes base mínimas
```dockerfile
# ❌ Imagen grande e innecesaria
FROM ubuntu:latest

# ✅ Imagen oficial y específica
FROM python:3.11-slim       # ~50MB en vez de ~700MB
FROM python:3.11-alpine     # ~20MB (atención: usa musl libc)
```

#### ✅ Especificar siempre versiones concretas
```dockerfile
# ❌ Versión no determinista
FROM node:latest

# ✅ Versión fija y reproducible
FROM node:20.11-alpine3.19
```

#### ✅ Optimizar el orden de capas para aprovechar la caché
```dockerfile
# ❌ Invalida caché con cualquier cambio en el código
COPY . .
RUN pip install -r requirements.txt

# ✅ La capa de dependencias se cachea si requirements.txt no cambia
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
```

#### ✅ Combinar comandos RUN para reducir capas
```dockerfile
# ❌ Crea 3 capas
RUN apt-get update
RUN apt-get install -y curl
RUN rm -rf /var/lib/apt/lists/*

# ✅ Una sola capa, más pequeña
RUN apt-get update && \
    apt-get install -y --no-install-recommends curl && \
    rm -rf /var/lib/apt/lists/*
```

#### ✅ Usar `.dockerignore`
```
# .dockerignore
.git/
.gitignore
node_modules/
__pycache__/
*.pyc
.env
.env.*
tests/
*.md
docs/
```

#### ✅ No ejecutar como root
```dockerfile
# ✅ Crear y usar usuario no privilegiado
RUN addgroup -S appgroup && adduser -S appuser -G appgroup
USER appuser
```

#### ✅ Usar HEALTHCHECK
```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
    CMD curl -f http://localhost:8080/health || exit 1
```

#### ✅ Un proceso por contenedor
El principio de responsabilidad única aplicado a contenedores: cada contenedor debe ejecutar **un solo proceso principal**. Esto facilita el escalado, el reinicio y el monitoreo.

### 15.2 Buenas prácticas de seguridad

| Práctica | Descripción |
|----------|-------------|
| No secrets en la imagen | Usar variables de entorno, Docker secrets o gestores de secretos (Vault, AWS Secrets Manager) |
| Escanear imágenes | Usar `docker scout`, Trivy, Snyk para buscar CVEs |
| Imágenes mínimas | Menos software = menor superficie de ataque |
| No usar `:latest` en producción | Puede cambiar inesperadamente |
| Actualizar imágenes base | Parchar vulnerabilidades del SO base |
| Leer-only filesystem | `docker run --read-only` cuando sea posible |
| Capabilities mínimas | `--cap-drop ALL --cap-add NET_BIND_SERVICE` |
| No montar el socket de Docker | `/var/run/docker.sock` da acceso root al host |

### 15.3 Limpieza del entorno

```bash
# Eliminar todos los recursos no usados (contenedores, imágenes, redes, volúmenes)
docker system prune

# Incluir volúmenes (cuidado: elimina datos persistentes!)
docker system prune --volumes

# Ver cuánto espacio usa Docker
docker system df
docker system df -v
```


---
## 16. Referencias

### Documentación Oficial

1. **Docker Documentation** — Documentación completa y oficial de Docker.  
   https://docs.docker.com/

2. **Docker Reference** — Referencia de la CLI, Dockerfile y APIs.  
   https://docs.docker.com/reference/

3. **Dockerfile Best Practices** — Guía oficial de buenas prácticas.  
   https://docs.docker.com/develop/develop-images/dockerfile_best-practices/

4. **Docker Compose Reference** — Referencia completa de Compose.  
   https://docs.docker.com/compose/compose-file/

5. **Docker Hub** — Registro de imágenes oficiales.  
   https://hub.docker.com/

### Libros

6. **"Docker Deep Dive"** — Nigel Poulton, 2024. Libro introductorio muy accesible.  
   ISBN: 978-1916585300

7. **"Docker: Up & Running"** — Sean P. Kane & Karl Matthias, O'Reilly, 3rd ed. 2023.  
   ISBN: 978-1098131814

8. **"The Docker Book"** — James Turnbull, 2023. Tutorial progresivo muy completo.  
   https://dockerbook.com/

9. **"Docker in Action"** — Jeff Nickoloff & Stephen Kuenzli, Manning, 2nd ed. 2019.  
   ISBN: 978-1617294761

### Artículos y Recursos Online

10. **"A Docker Tutorial for Beginners"** — docker-curriculum.com  
    https://docker-curriculum.com/

11. **"The Evolution of Linux Containers and Their Future"** — Red Hat Blog.  
    https://www.redhat.com/en/topics/containers/whats-a-linux-container

12. **"Docker Overview"** — Docker Official Docs.  
    https://docs.docker.com/get-started/overview/

13. **"Understanding Docker Networking"** — Docker Docs.  
    https://docs.docker.com/network/

14. **"Use volumes"** — Docker Docs.  
    https://docs.docker.com/storage/volumes/

15. **"Multi-stage builds"** — Docker Docs.  
    https://docs.docker.com/build/building/multi-stage/

16. **"OCI (Open Container Initiative)"** — Estándar de la industria para contenedores.  
    https://opencontainers.org/

17. **"cgroups and Linux Namespaces"** — Red Hat Developer.  
    https://www.redhat.com/en/topics/containers/whats-a-linux-namespace

18. **"Docker Security Cheat Sheet"** — OWASP.  
    https://cheatsheetseries.owasp.org/cheatsheets/Docker_Security_Cheat_Sheet.html

### Cursos y Laboratorios

19. **Play with Docker** — Laboratorio interactivo gratuito en el navegador.  
    https://labs.play-with-docker.com/

20. **Docker Labs** — Tutoriales y laboratorios de Docker Inc.  
    https://github.com/docker/labs

21. **"Docker Mastery"** — Bret Fisher (Udemy). Curso práctico muy completo.  
    https://www.udemy.com/course/docker-mastery/

22. **KodeKloud** — Plataforma con laboratorios interactivos de Docker y Kubernetes.  
    https://kodekloud.com/


---
## 17. Temas para la Clase 2

A continuación se presentan los temas ideales para continuar el aprendizaje de Docker en una **Clase 2**, ordenados de menor a mayor complejidad:

---

### 🔵 Bloque 1 — Profundización en Docker

1. **Docker Build Avanzado con BuildKit**
   - Caché avanzado (`--mount=type=cache`)
   - Heredoc en Dockerfile
   - Build secrets seguros (`--mount=type=secret`)
   - Construcción multiplataforma (`docker buildx`, `--platform linux/arm64`)
   - Exportación de artefactos sin imagen

2. **Docker Compose Avanzado**
   - Profiles (perfiles de servicios opcionales)
   - Extensiones con anchors YAML
   - Variables de entorno con `.env` files
   - Healthchecks y `condition: service_healthy`
   - Deploy configuration (réplicas, límites de recursos)

3. **Gestión avanzada de imágenes y capas**
   - Inspección profunda con `dive` (herramienta de análisis de capas)
   - Optimización de tamaño de imagen
   - Estrategias de multi-stage builds para Node.js, Java, Go, Rust
   - Distroless images (imágenes sin shell ni gestor de paquetes)

---

### 🟡 Bloque 2 — Seguridad en Docker

4. **Docker Security en profundidad**
   - Escaneo de vulnerabilidades con Docker Scout y Trivy
   - Linux capabilities y seccomp profiles
   - AppArmor y SELinux con Docker
   - Rootless Docker (ejecutar el daemon sin privilegios root)
   - Namespaces de usuario (user namespace remapping)
   - Firmas de imágenes con Cosign y Notary v2

5. **Gestión de secretos**
   - Docker Secrets (en Swarm)
   - Integración con HashiCorp Vault
   - Integración con AWS Secrets Manager / Azure Key Vault
   - Build secrets con BuildKit

---

### 🟠 Bloque 3 — Docker en el ciclo de vida de desarrollo (DevOps)

6. **Docker en CI/CD**
   - Integración con GitHub Actions: build, test y push de imágenes
   - Integración con GitLab CI
   - Docker-in-Docker (DinD) vs Docker socket binding
   - Caché de layers en pipelines CI
   - Matrices de build para múltiples versiones

7. **Testing con Docker**
   - Testcontainers: levantamiento de dependencias reales en tests
   - Docker Compose para entornos de integración
   - Estrategia de imágenes para tests (imágenes de test vs producción)

8. **Monitoreo y logging de contenedores**
   - Logging drivers (json-file, syslog, fluentd, gelf)
   - Integración con ELK Stack (Elasticsearch, Logstash, Kibana)
   - Monitoreo con Prometheus y cAdvisor
   - Integración con Grafana

---

### 🔴 Bloque 4 — Orquestación y producción

9. **Docker Swarm**
   - Qué es la orquestación de contenedores
   - Inicializar un clúster Swarm
   - Servicios, réplicas y escalado
   - Rolling updates y rollbacks
   - Stacks en Swarm
   - Cuándo usar Swarm vs Kubernetes

10. **Introducción a Kubernetes (contexto desde Docker)**
    - De contenedores a pods
    - Comparación Docker Compose vs Kubernetes manifests
    - Kompose: convertir docker-compose.yml a manifiestos Kubernetes
    - Herramientas de desarrollo local: Minikube, Kind, k3d

11. **Registro privado de imágenes**
    - Desplegar Harbor en producción
    - Políticas de retención y limpieza de imágenes
    - Integración con LDAP/OIDC
    - Replicación entre registros

---

### 🟣 Bloque 5 — Casos de uso especializados

12. **Docker para Data Science y Machine Learning**
    - Imágenes con GPU (NVIDIA Container Toolkit)
    - Jupyter Notebooks en contenedores
    - MLflow con Docker
    - Packaging de modelos con Docker

13. **Docker para desarrollo con bases de datos**
    - PostgreSQL, MySQL, MongoDB en contenedores
    - Inicialización con scripts SQL
    - Backup y restore de bases de datos containerizadas
    - Estrategias de migración de esquemas

14. **Performance y optimización**
    - Profiling de contenedores
    - Benchmarking de overhead de contenedores
    - Tuning de networking (MTU, buffer sizes)
    - Storage drivers: OverlayFS vs otros

---

> 💡 **Recomendación de secuencia para Clase 2:** Comenzar con los Bloques 1 y 2 para consolidar las bases, luego el Bloque 3 para conectar con prácticas DevOps modernas, y dejar los Bloques 4 y 5 para sesiones posteriores según los intereses del grupo.
